# Audio-Only vs Full Model: Virality Prediction Comparison

This notebook builds two contrasting ensemble models:
- **Audio-Only Model**: Spotify + Librosa features only (scientifically interesting, no confounders)
- **Full Model**: All features including channel metadata, tags, upload date (higher accuracy but potential confounders)

**Critical Target Leakage Prevention**:
- Drops features used to construct the viral target: view_count, like_count, comment_count, like_rate, comment_rate, virality_score
- This ensures we're building models that can predict virality BEFORE a song is released

## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import joblib
from pathlib import Path

# Preprocessing
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score

# Models
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import VotingClassifier, StackingClassifier

# Metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve, auc
)

warnings.filterwarnings('ignore')
sns.set_style('darkgrid')

# Set random seed
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("All libraries imported successfully!")

## 2. Load Combined Features Dataset

In [ ]:
# Load combined features dataset
DATA_PATH = "../data/processed/combined_features_cleaned.csv"
df = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {df.shape}")
print(f"\nColumn names ({len(df.columns)} total):")
print(df.columns.tolist())
print(f"\nData types:\n{df.dtypes}")
print(f"\nMissing values:")
print(df.isnull().sum())
print(f"\nClass distribution (viral):\n{df['viral'].value_counts()}")

## 3. Feature Selection and Target Leakage Prevention

### 3.1 Identify Feature Categories and Remove Leakage Features

In [ ]:
# Define leakage features (used to construct viral target or engagement metrics)
LEAKAGE_FEATURES = [
    'view_count', 'like_count', 'comment_count',  # Engagement metrics
    'like_rate', 'comment_rate',  # Computed engagement rates
    'virality_score'  # Target construct
]

# Define Spotify audio features (manually list known Spotify audio features)
SPOTIFY_AUDIO_FEATURES = [
    'danceability', 'energy', 'valence', 'acousticness',
    'instrumentalness', 'speechiness', 'liveness', 'loudness', 'tempo'
]

# Define identifiers and metadata (to be excluded from features)
ID_AND_METADATA = ['track_id', 'track_name', 'artists', 'viral']

# Librosa features include spectral, MFCC, chroma, etc. 
# (anything that starts with known prefixes)
LIBROSA_PREFIXES = [
    'spectral_', 'mfcc_', 'chroma_', 'zcr_', 'rms_', 'onset_', 'tonnetz_', 'tempo_librosa'
]

# Identify all Librosa features
librosa_features = [col for col in df.columns 
                    if any(col.startswith(prefix) for prefix in LIBROSA_PREFIXES)]

# Create feature groups
all_features = set(df.columns) - set(ID_AND_METADATA) - set(LEAKAGE_FEATURES)
audio_only_features = set(SPOTIFY_AUDIO_FEATURES) | set(librosa_features)
full_model_features = list(all_features)
audio_only_features = list(audio_only_features & all_features)

print(f"Total features available: {len(all_features)}")
print(f"\nLeakage features (REMOVED): {LEAKAGE_FEATURES}")
print(f"\nAudio-only features ({len(audio_only_features)}): ")
print(f"  - Spotify: {[f for f in SPOTIFY_AUDIO_FEATURES if f in audio_only_features]}")
print(f"  - Librosa: {librosa_features[:5]}... ({len(librosa_features)} total)")
print(f"\nFull model features ({len(full_model_features)})")
print(f"  - Includes all audio + metadata (but excludes leakage)")

# Target variable
y = df['viral'].copy()
print(f"\nTarget class distribution:\n{y.value_counts()}")

### 3.2 Prepare Audio-Only Feature Set

In [ ]:
# Audio-only: Spotify + Librosa features only
X_audio = df[audio_only_features].copy()

print(f"Audio-Only Feature Matrix:")
print(f"  Shape: {X_audio.shape}")
print(f"  Features: {X_audio.columns.tolist()}")
print(f"  Missing values: {X_audio.isnull().sum().sum()}")
print(f"  Data types:\n{X_audio.dtypes}")

# Check for categorical features and encode if necessary
categorical_cols_audio = X_audio.select_dtypes(include=['object']).columns.tolist()
if categorical_cols_audio:
    print(f"\n  Categorical features to encode: {categorical_cols_audio}")
    le_dict_audio = {}
    X_audio_encoded = X_audio.copy()
    for col in categorical_cols_audio:
        le = LabelEncoder()
        X_audio_encoded[col] = le.fit_transform(X_audio_encoded[col].astype(str))
        le_dict_audio[col] = le
    X_audio = X_audio_encoded
else:
    print(f"\n  No categorical features found")

### 3.3 Prepare Full Feature Set

In [ ]:
# Full model: Everything except leakage and identifiers
X_full = df[full_model_features].copy()

print(f"Full Model Feature Matrix:")
print(f"  Shape: {X_full.shape}")
print(f"  Missing values: {X_full.isnull().sum().sum()}")

# Identify and encode categorical features in full model
categorical_cols_full = X_full.select_dtypes(include=['object']).columns.tolist()
print(f"\n  Categorical features to encode: {categorical_cols_full}")

le_dict_full = {}
X_full_encoded = X_full.copy()
for col in categorical_cols_full:
    le = LabelEncoder()
    X_full_encoded[col] = le.fit_transform(X_full_encoded[col].astype(str))
    le_dict_full[col] = le
    print(f"    - Encoded {col}: {len(le.classes_)} unique values")

X_full = X_full_encoded

print(f"\nFull model final shape: {X_full.shape}")
print(f"Audio-only model final shape: {X_audio.shape}")

## 4. Train-Test Split and Scaling

### 4.1 Split Audio-Only Dataset

In [ ]:
# Audio-only split
X_audio_train, X_audio_test, y_audio_train, y_audio_test = train_test_split(
    X_audio, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

# Scale audio-only features
scaler_audio = StandardScaler()
X_audio_train_scaled = scaler_audio.fit_transform(X_audio_train)
X_audio_test_scaled = scaler_audio.transform(X_audio_test)

X_audio_train_scaled = pd.DataFrame(X_audio_train_scaled, columns=X_audio.columns, index=X_audio_train.index)
X_audio_test_scaled = pd.DataFrame(X_audio_test_scaled, columns=X_audio.columns, index=X_audio_test.index)

print(f"Audio-Only Model:")
print(f"  Training set: {X_audio_train_scaled.shape}")
print(f"  Test set: {X_audio_test_scaled.shape}")
print(f"  Train viral distribution: {dict(y_audio_train.value_counts())}")
print(f"  Test viral distribution: {dict(y_audio_test.value_counts())}")

### 4.2 Split Full Model Dataset

In [ ]:
# Full model split (using same indices as audio for fair comparison)
X_full_train, X_full_test, y_full_train, y_full_test = train_test_split(
    X_full, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

# Scale full model features
scaler_full = StandardScaler()
X_full_train_scaled = scaler_full.fit_transform(X_full_train)
X_full_test_scaled = scaler_full.transform(X_full_test)

X_full_train_scaled = pd.DataFrame(X_full_train_scaled, columns=X_full.columns, index=X_full_train.index)
X_full_test_scaled = pd.DataFrame(X_full_test_scaled, columns=X_full.columns, index=X_full_test.index)

print(f"Full Model:")
print(f"  Training set: {X_full_train_scaled.shape}")
print(f"  Test set: {X_full_test_scaled.shape}")
print(f"  Train viral distribution: {dict(y_full_train.value_counts())}")
print(f"  Test viral distribution: {dict(y_full_test.value_counts())}")

# Setup cross-validation
cv_split = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

## 5. Build Base Learner and Ensemble Models

### 5.1 Audio-Only Models

In [ ]:
# Build audio-only base learners and ensemble
print("="*80)
print("BUILDING AUDIO-ONLY MODELS (Scientifically Interesting)")
print("="*80)

# XGBoost
audio_xgb = XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.1, 
                          random_state=RANDOM_STATE, verbosity=0, eval_metric='logloss')
audio_xgb.fit(X_audio_train_scaled, y_audio_train)
audio_xgb_cv = cross_val_score(audio_xgb, X_audio_train_scaled, y_audio_train, cv=cv_split, scoring='f1_macro')
print(f"Audio XGBoost CV F1-Macro: {audio_xgb_cv.mean():.4f} (+/- {audio_xgb_cv.std():.4f})")

# LightGBM
audio_lgb = LGBMClassifier(n_estimators=300, max_depth=8, learning_rate=0.1, 
                           random_state=RANDOM_STATE, verbose=-1)
audio_lgb.fit(X_audio_train_scaled, y_audio_train)
audio_lgb_cv = cross_val_score(audio_lgb, X_audio_train_scaled, y_audio_train, cv=cv_split, scoring='f1_macro')
print(f"Audio LightGBM CV F1-Macro: {audio_lgb_cv.mean():.4f} (+/- {audio_lgb_cv.std():.4f})")

# CatBoost
audio_catboost = CatBoostClassifier(iterations=300, depth=6, learning_rate=0.1, 
                                    random_state=RANDOM_STATE, verbose=0)
audio_catboost.fit(X_audio_train_scaled, y_audio_train)
audio_catboost_cv = cross_val_score(audio_catboost, X_audio_train_scaled, y_audio_train, cv=cv_split, scoring='f1_macro')
print(f"Audio CatBoost CV F1-Macro: {audio_catboost_cv.mean():.4f} (+/- {audio_catboost_cv.std():.4f})")

# Audio Voting Classifier
audio_voting = VotingClassifier(
    estimators=[
        ('xgb', XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.1, 
                             random_state=RANDOM_STATE, verbosity=0, eval_metric='logloss')),
        ('lgb', LGBMClassifier(n_estimators=300, max_depth=8, learning_rate=0.1, 
                             random_state=RANDOM_STATE, verbose=-1)),
        ('catboost', CatBoostClassifier(iterations=300, depth=6, learning_rate=0.1, 
                                       random_state=RANDOM_STATE, verbose=0))
    ],
    voting='soft',
    n_jobs=-1
)
audio_voting.fit(X_audio_train_scaled, y_audio_train)
audio_voting_cv = cross_val_score(audio_voting, X_audio_train_scaled, y_audio_train, cv=cv_split, scoring='f1_macro')
print(f"Audio Voting Classifier CV F1-Macro: {audio_voting_cv.mean():.4f} (+/- {audio_voting_cv.std():.4f})")

# Audio Stacking Classifier
audio_stacking = StackingClassifier(
    estimators=[
        ('xgb', XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.1, 
                             random_state=RANDOM_STATE, verbosity=0, eval_metric='logloss')),
        ('lgb', LGBMClassifier(n_estimators=300, max_depth=8, learning_rate=0.1, 
                             random_state=RANDOM_STATE, verbose=-1)),
        ('catboost', CatBoostClassifier(iterations=300, depth=6, learning_rate=0.1, 
                                       random_state=RANDOM_STATE, verbose=0))
    ],
    final_estimator=LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    passthrough=True,
    cv=cv_split,
    n_jobs=-1
)
audio_stacking.fit(X_audio_train_scaled, y_audio_train)
audio_stacking_cv = cross_val_score(audio_stacking, X_audio_train_scaled, y_audio_train, cv=cv_split, scoring='f1_macro')
print(f"Audio Stacking Classifier CV F1-Macro: {audio_stacking_cv.mean():.4f} (+/- {audio_stacking_cv.std():.4f})")

print("\nAudio-only models training complete!")

### 5.2 Full Models (All Features)

In [ ]:
print("\n" + "="*80)
print("BUILDING FULL MODELS (High Accuracy, Includes Metadata)")
print("="*80)

# XGBoost
full_xgb = XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.1, 
                         random_state=RANDOM_STATE, verbosity=0, eval_metric='logloss')
full_xgb.fit(X_full_train_scaled, y_full_train)
full_xgb_cv = cross_val_score(full_xgb, X_full_train_scaled, y_full_train, cv=cv_split, scoring='f1_macro')
print(f"Full XGBoost CV F1-Macro: {full_xgb_cv.mean():.4f} (+/- {full_xgb_cv.std():.4f})")

# LightGBM
full_lgb = LGBMClassifier(n_estimators=300, max_depth=8, learning_rate=0.1, 
                          random_state=RANDOM_STATE, verbose=-1)
full_lgb.fit(X_full_train_scaled, y_full_train)
full_lgb_cv = cross_val_score(full_lgb, X_full_train_scaled, y_full_train, cv=cv_split, scoring='f1_macro')
print(f"Full LightGBM CV F1-Macro: {full_lgb_cv.mean():.4f} (+/- {full_lgb_cv.std():.4f})")

# CatBoost
full_catboost = CatBoostClassifier(iterations=300, depth=6, learning_rate=0.1, 
                                   random_state=RANDOM_STATE, verbose=0)
full_catboost.fit(X_full_train_scaled, y_full_train)
full_catboost_cv = cross_val_score(full_catboost, X_full_train_scaled, y_full_train, cv=cv_split, scoring='f1_macro')
print(f"Full CatBoost CV F1-Macro: {full_catboost_cv.mean():.4f} (+/- {full_catboost_cv.std():.4f})")

# Full Voting Classifier
full_voting = VotingClassifier(
    estimators=[
        ('xgb', XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.1, 
                             random_state=RANDOM_STATE, verbosity=0, eval_metric='logloss')),
        ('lgb', LGBMClassifier(n_estimators=300, max_depth=8, learning_rate=0.1, 
                             random_state=RANDOM_STATE, verbose=-1)),
        ('catboost', CatBoostClassifier(iterations=300, depth=6, learning_rate=0.1, 
                                       random_state=RANDOM_STATE, verbose=0))
    ],
    voting='soft',
    n_jobs=-1
)
full_voting.fit(X_full_train_scaled, y_full_train)
full_voting_cv = cross_val_score(full_voting, X_full_train_scaled, y_full_train, cv=cv_split, scoring='f1_macro')
print(f"Full Voting Classifier CV F1-Macro: {full_voting_cv.mean():.4f} (+/- {full_voting_cv.std():.4f})")

# Full Stacking Classifier
full_stacking = StackingClassifier(
    estimators=[
        ('xgb', XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.1, 
                             random_state=RANDOM_STATE, verbosity=0, eval_metric='logloss')),
        ('lgb', LGBMClassifier(n_estimators=300, max_depth=8, learning_rate=0.1, 
                             random_state=RANDOM_STATE, verbose=-1)),
        ('catboost', CatBoostClassifier(iterations=300, depth=6, learning_rate=0.1, 
                                       random_state=RANDOM_STATE, verbose=0))
    ],
    final_estimator=LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    passthrough=True,
    cv=cv_split,
    n_jobs=-1
)
full_stacking.fit(X_full_train_scaled, y_full_train)
full_stacking_cv = cross_val_score(full_stacking, X_full_train_scaled, y_full_train, cv=cv_split, scoring='f1_macro')
print(f"Full Stacking Classifier CV F1-Macro: {full_stacking_cv.mean():.4f} (+/- {full_stacking_cv.std():.4f})")

print("\nFull models training complete!")

## 6. Evaluation and Model Comparison

### 6.1 Evaluation Function

In [ ]:
def evaluate_model(model, X_test, y_test, model_name):
    """Evaluate a model and return metrics."""
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    metrics = {
        'Model': model_name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1-Macro': f1_score(y_test, y_pred, average='macro', zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, y_pred_proba)
    }
    return metrics, y_pred, y_pred_proba

# Evaluate Audio-Only Models
print("\n" + "="*80)
print("AUDIO-ONLY MODEL RESULTS (Test Set)")
print("="*80)

audio_results = []
audio_predictions = {}

audio_metrics_xgb, audio_pred_xgb, audio_prob_xgb = evaluate_model(audio_xgb, X_audio_test_scaled, y_audio_test, 'Audio XGBoost')
audio_results.append(audio_metrics_xgb)
audio_predictions['XGBoost'] = (audio_pred_xgb, audio_prob_xgb)

audio_metrics_lgb, audio_pred_lgb, audio_prob_lgb = evaluate_model(audio_lgb, X_audio_test_scaled, y_audio_test, 'Audio LightGBM')
audio_results.append(audio_metrics_lgb)
audio_predictions['LightGBM'] = (audio_pred_lgb, audio_prob_lgb)

audio_metrics_catboost, audio_pred_catboost, audio_prob_catboost = evaluate_model(audio_catboost, X_audio_test_scaled, y_audio_test, 'Audio CatBoost')
audio_results.append(audio_metrics_catboost)
audio_predictions['CatBoost'] = (audio_pred_catboost, audio_prob_catboost)

audio_metrics_voting, audio_pred_voting, audio_prob_voting = evaluate_model(audio_voting, X_audio_test_scaled, y_audio_test, 'Audio Voting')
audio_results.append(audio_metrics_voting)
audio_predictions['Voting'] = (audio_pred_voting, audio_prob_voting)

audio_metrics_stacking, audio_pred_stacking, audio_prob_stacking = evaluate_model(audio_stacking, X_audio_test_scaled, y_audio_test, 'Audio Stacking')
audio_results.append(audio_metrics_stacking)
audio_predictions['Stacking'] = (audio_pred_stacking, audio_prob_stacking)

audio_results_df = pd.DataFrame(audio_results)
print(audio_results_df.to_string(index=False))

# Evaluate Full Models
print("\n" + "="*80)
print("FULL MODEL RESULTS (Test Set)")
print("="*80)

full_results = []
full_predictions = {}

full_metrics_xgb, full_pred_xgb, full_prob_xgb = evaluate_model(full_xgb, X_full_test_scaled, y_full_test, 'Full XGBoost')
full_results.append(full_metrics_xgb)
full_predictions['XGBoost'] = (full_pred_xgb, full_prob_xgb)

full_metrics_lgb, full_pred_lgb, full_prob_lgb = evaluate_model(full_lgb, X_full_test_scaled, y_full_test, 'Full LightGBM')
full_results.append(full_metrics_lgb)
full_predictions['LightGBM'] = (full_pred_lgb, full_prob_lgb)

full_metrics_catboost, full_pred_catboost, full_prob_catboost = evaluate_model(full_catboost, X_full_test_scaled, y_full_test, 'Full CatBoost')
full_results.append(full_metrics_catboost)
full_predictions['CatBoost'] = (full_pred_catboost, full_prob_catboost)

full_metrics_voting, full_pred_voting, full_prob_voting = evaluate_model(full_voting, X_full_test_scaled, y_full_test, 'Full Voting')
full_results.append(full_metrics_voting)
full_predictions['Voting'] = (full_pred_voting, full_prob_voting)

full_metrics_stacking, full_pred_stacking, full_prob_stacking = evaluate_model(full_stacking, X_full_test_scaled, y_full_test, 'Full Stacking')
full_results.append(full_metrics_stacking)
full_predictions['Stacking'] = (full_pred_stacking, full_prob_stacking)

full_results_df = pd.DataFrame(full_results)
print(full_results_df.to_string(index=False))

### 6.2 Direct Model Comparison: Audio-Only vs Full

In [ ]:
# Compare best models from each approach
audio_best_idx = audio_results_df['F1-Macro'].idxmax()
audio_best_model = audio_results_df.loc[audio_best_idx]

full_best_idx = full_results_df['F1-Macro'].idxmax()
full_best_model = full_results_df.loc[full_best_idx]

print("\n" + "="*80)
print("COMPARISON: BEST AUDIO-ONLY vs BEST FULL MODEL")
print("="*80)

comparison_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Macro', 'ROC-AUC'],
    'Best Audio-Only': [
        audio_best_model['Accuracy'],
        audio_best_model['Precision'],
        audio_best_model['Recall'],
        audio_best_model['F1-Macro'],
        audio_best_model['ROC-AUC']
    ],
    'Best Full': [
        full_best_model['Accuracy'],
        full_best_model['Precision'],
        full_best_model['Recall'],
        full_best_model['F1-Macro'],
        full_best_model['ROC-AUC']
    ]
})

comparison_df['Difference (Full - Audio)'] = comparison_df['Best Full'] - comparison_df['Best Audio-Only']
comparison_df['% Improvement'] = (comparison_df['Difference (Full - Audio)'] / comparison_df['Best Audio-Only'] * 100).round(2)

print(comparison_df.to_string(index=False))

print(f"\n{'='*80}")
print("BEST MODELS SUMMARY")
print(f"{'='*80}")
print(f"Audio-Only: {audio_best_model['Model']}")
print(f"  - F1-Macro: {audio_best_model['F1-Macro']:.4f}")
print(f"  - Accuracy: {audio_best_model['Accuracy']:.4f}")
print(f"  - ROC-AUC: {audio_best_model['ROC-AUC']:.4f}")

print(f"\nFull: {full_best_model['Model']}")
print(f"  - F1-Macro: {full_best_model['F1-Macro']:.4f}")
print(f"  - Accuracy: {full_best_model['Accuracy']:.4f}")
print(f"  - ROC-AUC: {full_best_model['ROC-AUC']:.4f}")

print(f"\nKey Insight:")
print(f"  - Full model F1-Macro {comparison_df.loc[3, 'Difference (Full - Audio)']:.4f} points higher")
print(f"  - This {comparison_df.loc[3, '% Improvement']:.1f}% improvement comes from metadata/channel effects")
print(f"\nFor Real-World Use Case (before release):")
print(f"  → Use AUDIO-ONLY model ({audio_best_model['Model']})")
print(f"  → No channel metadata available at prediction time")
print(f"\nFor Accuracy Benchmarking:")
print(f"  → Full model shows potential of all available signals")

### 6.3 Confusion Matrices Comparison

In [ ]:
# Plot confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Audio best model confusion matrix
audio_best_name = audio_best_model['Model'].replace('Audio ', '')
audio_cm = confusion_matrix(y_audio_test, audio_predictions[audio_best_name][0])
sns.heatmap(audio_cm, annot=True, fmt='d', cmap='Blues', ax=axes[0], cbar=False)
axes[0].set_title(f'Audio-Only Model\n({audio_best_model["Model"]})')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')
axes[0].set_xticklabels(['Non-Viral', 'Viral'])
axes[0].set_yticklabels(['Non-Viral', 'Viral'])

# Full best model confusion matrix
full_best_name = full_best_model['Model'].replace('Full ', '')
full_cm = confusion_matrix(y_full_test, full_predictions[full_best_name][0])
sns.heatmap(full_cm, annot=True, fmt='d', cmap='Greens', ax=axes[1], cbar=False)
axes[1].set_title(f'Full Model\n({full_best_model["Model"]})')
axes[1].set_ylabel('Actual')
axes[1].set_xlabel('Predicted')
axes[1].set_xticklabels(['Non-Viral', 'Viral'])
axes[1].set_yticklabels(['Non-Viral', 'Viral'])

plt.tight_layout()
plt.savefig('../models/audio_vs_full_confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()
print("Confusion matrices saved!")

### 6.4 ROC Curves: Audio-Only vs Full

In [ ]:
# Plot ROC curves
fig, ax = plt.subplots(figsize=(12, 8))

# Audio-only ROC curve
audio_fpr, audio_tpr, _ = roc_curve(y_audio_test, audio_predictions[audio_best_name][1])
audio_auc = auc(audio_fpr, audio_tpr)
ax.plot(audio_fpr, audio_tpr, label=f'Audio-Only ({audio_best_model["Model"]}, AUC={audio_auc:.3f})', 
        color='blue', linewidth=2)

# Full model ROC curve
full_fpr, full_tpr, _ = roc_curve(y_full_test, full_predictions[full_best_name][1])
full_auc = auc(full_fpr, full_tpr)
ax.plot(full_fpr, full_tpr, label=f'Full Model ({full_best_model["Model"]}, AUC={full_auc:.3f})', 
        color='green', linewidth=2)

# Random classifier
ax.plot([0, 1], [0, 1], 'k--', linewidth=1.5, label='Random Classifier')

ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves: Audio-Only vs Full Model Comparison', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../models/audio_vs_full_roc_curves.png', dpi=300, bbox_inches='tight')
plt.show()
print("ROC curves saved!")

### 6.5 Feature Importance: Top Features from Full Model

In [ ]:
# Get feature importance from both models
# Audio model (XGBoost or similar)
if hasattr(audio_xgb, 'feature_importances_'):
    audio_importance = pd.DataFrame({
        'feature': X_audio.columns,
        'importance': audio_xgb.feature_importances_
    }).sort_values('importance', ascending=False).head(15)

# Full model (XGBoost)
full_importance = pd.DataFrame({
    'feature': X_full.columns,
    'importance': full_xgb.feature_importances_
}).sort_values('importance', ascending=False).head(15)

# Plot feature importances
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Audio importance
audio_importance_sorted = audio_importance.sort_values('importance')
axes[0].barh(range(len(audio_importance_sorted)), audio_importance_sorted['importance'].values, color='skyblue')
axes[0].set_yticks(range(len(audio_importance_sorted)))
axes[0].set_yticklabels(audio_importance_sorted['feature'].values, fontsize=9)
axes[0].set_xlabel('Importance', fontsize=11)
axes[0].set_title('Audio-Only Model\nTop 15 Feature Importances', fontsize=12, fontweight='bold')
axes[0].grid(axis='x', alpha=0.3)

# Full importance
full_importance_sorted = full_importance.sort_values('importance')
axes[1].barh(range(len(full_importance_sorted)), full_importance_sorted['importance'].values, color='lightgreen')
axes[1].set_yticks(range(len(full_importance_sorted)))
axes[1].set_yticklabels(full_importance_sorted['feature'].values, fontsize=9)
axes[1].set_xlabel('Importance', fontsize=11)
axes[1].set_title('Full Model\nTop 15 Feature Importances', fontsize=12, fontweight='bold')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('../models/audio_vs_full_feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()
print("Feature importance plot saved!")

print("\nTop 15 Features in Audio-Only Model:")
print(audio_importance.to_string(index=False))

print("\nTop 15 Features in Full Model:")
print(full_importance.to_string(index=False))

## 7. Save Best Models and Artifacts

In [ ]:
# Create models directory
models_dir = Path('../models')
models_dir.mkdir(parents=True, exist_ok=True)

# Extract actual model objects from best models
audio_best_models_map = {
    'Audio XGBoost': audio_xgb,
    'Audio LightGBM': audio_lgb,
    'Audio CatBoost': audio_catboost,
    'Audio Voting': audio_voting,
    'Audio Stacking': audio_stacking
}

full_best_models_map = {
    'Full XGBoost': full_xgb,
    'Full LightGBM': full_lgb,
    'Full CatBoost': full_catboost,
    'Full Voting': full_voting,
    'Full Stacking': full_stacking
}

# Save best audio model for production use (recommended before release)
audio_best_model_obj = audio_best_models_map[audio_best_model['Model']]
audio_save_path = models_dir / 'audio_only_best_model.pkl'
joblib.dump(audio_best_model_obj, audio_save_path)
print(f"✓ Best Audio-Only Model saved: {audio_best_model['Model']}")
print(f"  - Path: {audio_save_path}")
print(f"  - F1-Macro: {audio_best_model['F1-Macro']:.4f}")

# Save best full model (reference benchmark)
full_best_model_obj = full_best_models_map[full_best_model['Model']]
full_save_path = models_dir / 'full_best_model.pkl'
joblib.dump(full_best_model_obj, full_save_path)
print(f"\n✓ Best Full Model saved: {full_best_model['Model']}")
print(f"  - Path: {full_save_path}")
print(f"  - F1-Macro: {full_best_model['F1-Macro']:.4f}")

# Save scalers
scaler_audio_path = models_dir / 'audio_only_scaler.pkl'
joblib.dump(scaler_audio, scaler_audio_path)
print(f"\n✓ Audio-Only Scaler saved")

scaler_full_path = models_dir / 'full_scaler.pkl'
joblib.dump(scaler_full, scaler_full_path)
print(f"✓ Full Model Scaler saved")

# Save label encoders
encoder_full_path = models_dir / 'full_label_encoders.pkl'
joblib.dump(le_dict_full, encoder_full_path)
print(f"✓ Lab Encoders (Full) saved")

# Save results dataframes
audio_results_path = models_dir / 'audio_only_results.csv'
audio_results_df.to_csv(audio_results_path, index=False)
print(f"✓ Audio-Only Results saved")

full_results_path = models_dir / 'full_results.csv'
full_results_df.to_csv(full_results_path, index=False)
print(f"✓ Full Results saved")

# Save comparison dataframe
comparison_path = models_dir / 'audio_vs_full_comparison.csv'
comparison_df.to_csv(comparison_path, index=False)
print(f"✓ Comparison DataFrame saved")

print(f"\nAll models and artifacts saved to {models_dir}/")

## 8. Final Summary and Recommendations

In [ ]:
print("\n" + "="*100)
print("AUDIO-ONLY vs FULL MODEL COMPARISON - EXECUTIVE SUMMARY")
print("="*100)

print(f"\n📊 DATASET & FEATURES:")
print(f"  • Total samples: {len(df):,}")
print(f"  • Features (after removing leakage): {X_full.shape[1]}")
print(f"  • Audio-only features: {X_audio.shape[1]} (Spotify + Librosa)")
print(f"  • Additional features in full model: {X_full.shape[1] - X_audio.shape[1]} (metadata, tags, upload date, etc.)")

print(f"\n🎯 KEY FINDINGS:")
print(f"\n  1. AUDIO-ONLY MODEL (Scientifically Interesting)")
print(f"     Model: {audio_best_model['Model']}")
print(f"     └─ F1-Macro: {audio_best_model['F1-Macro']:.4f}")
print(f"     └─ Accuracy: {audio_best_model['Accuracy']:.4f}")
print(f"     └─ ROC-AUC: {audio_best_model['ROC-AUC']:.4f}")
print(f"\n  2. FULL MODEL (Highest Accuracy Benchmark)")
print(f"     Model: {full_best_model['Model']}")
print(f"     └─ F1-Macro: {full_best_model['F1-Macro']:.4f}")
print(f"     └─ Accuracy: {full_best_model['Accuracy']:.4f}")
print(f"     └─ ROC-AUC: {full_best_model['ROC-AUC']:.4f}")

print(f"\n  3. PERFORMANCE GAP:")
print(f"     F1-Macro improvement (Full vs Audio): {comparison_df.loc[3, 'Difference (Full - Audio)']:.4f}")
print(f"     Relative improvement: +{comparison_df.loc[3, '% Improvement']:.1f}%")

print(f"\n⚠️  TARGET LEAKAGE PREVENTION (Critical):")
print(f"  ✓ Removed leakage features: {LEAKAGE_FEATURES}")
print(f"  ✓ These features (view_count, like_count, comment_count) are used")
print(f"    to construct the viral target and are NOT available at prediction time")

print(f"\n🚀 RECOMMENDATIONS FOR REAL-WORLD DEPLOYMENT:")
print(f"\n  USE CASE 1: Predict virality BEFORE song release")
print(f"  ────────────────────────────────────────────")
print(f"  → Recommendation: AUDIO-ONLY Model ({audio_best_model['Model']})")
print(f"  → Why: Channel metadata unavailable before release")
print(f"        Audio features capture song's intrinsic viability")
print(f"  → Expected F1-Macro: {audio_best_model['F1-Macro']:.4f}")

print(f"\n  USE CASE 2: Benchmark maximum accuracy potential")
print(f"  ────────────────────────────────────────────")
print(f"  → Recommendation: FULL Model ({full_best_model['Model']})")
print(f"  → Why: Shows how much channel + metadata improve predictions")
print(f"  → Expected F1-Macro: {full_best_model['F1-Macro']:.4f}")

print(f"\n  USE CASE 3: Research & Impact Analysis")
print(f"  ────────────────────────────────────────────")
print(f"  → Compare both models to quantify channel effects")
print(f"  → Gap analysis: {comparison_df.loc[3, '% Improvement']:.1f}% improvement from metadata")
print(f"  → Suggests: Channel follower count is a {comparison_df.loc[3, '% Improvement']:.1f}% confounder")

print(f"\n📁 SAVED ARTIFACTS:")
print(f"  ✓ audio_only_best_model.pkl - Production ready (recommended)")
print(f"  ✓ full_best_model.pkl - Reference benchmark")
print(f"  ✓ audio_only_scaler.pkl, full_scaler.pkl")
print(f"  ✓ audio_only_results.csv, full_results.csv")
print(f"  ✓ audio_vs_full_comparison.csv")
print(f"  ✓ Visualizations: confusion matrices, ROC curves, feature importance")

print(f"\n{'='*100}\n")
